# TRELLIS trial: one picture in, a 3D model out

The code lives in the RoomPitch repo, in `notebooks/trellis/trellis_trial.py`. This notebook pulls that folder and calls it, so after a change is pushed, just **re-run cell 0**; no re-import needed.

**Settings panel:** Accelerator **GPU T4 x2**, Internet **on**. If the repo is private, add a GitHub token under Add-ons → Secrets named `GITHUB_TOKEN` (fine-grained, read-only "Contents" on this repo) and tick it for this notebook.

Run cells 0 to 2 once per session (the install takes about 10 to 15 minutes); then cells 3 to 6 for each image. Results go to `/kaggle/working/out/` (the Output tab).

## 0 · Pull the code (re-run after every push)

In [ ]:
REPO, BRANCH = "MohamedMohsen95/roompitch_Demo", "staging"

import base64, importlib, os, subprocess, sys
SRC = "/tmp/roompitch"                       # outside /kaggle/working, so the clone isn't saved with the notebook
auth = []
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    # the token goes in a header for each git call: never in the URL or .git/config
    auth = ["-c", "http.extraHeader=Authorization: Basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()]
except Exception:
    pass                                     # no secret: fine for a public repo

def git(*args, cwd=None):
    p = subprocess.run(["git", *auth, *args], cwd=cwd, capture_output=True, text=True)
    if p.returncode:
        raise RuntimeError(f"git {args[0]} failed (private repo without a GITHUB_TOKEN secret?):\n{p.stderr[-1500:]}")
    return p.stdout.strip()

if not os.path.isdir(f"{SRC}/.git"):
    git("clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", "-b", BRANCH, f"https://github.com/{REPO}.git", SRC)
    git("sparse-checkout", "set", "notebooks/trellis", cwd=SRC)
else:
    git("fetch", "-q", "--depth", "1", "origin", BRANCH, cwd=SRC)
    git("reset", "-q", "--hard", "FETCH_HEAD", cwd=SRC)
print("Code at", git("log", "-1", "--format=%h %s (%cr)", cwd=SRC))

if f"{SRC}/notebooks/trellis" not in sys.path:
    sys.path.insert(0, f"{SRC}/notebooks/trellis")
import trellis_trial as tt
tt = importlib.reload(tt)

## 1 · Install (once per session, about 10–15 min)

In [ ]:
tt.install()

## 2 · Load the model (once per session, about 1–2 min)

In [ ]:
pipe = tt.load()

## 3 · Choose the image

One object on a plain background works best; the background is removed for you. The first one set wins: an upload with the button, `IMAGE_URL`, `IMAGE_PATH` (e.g. an attached dataset under `/kaggle/input/...`), then `SAMPLE`.

In [ ]:
IMAGE_URL  = ""
IMAGE_PATH = ""
SAMPLE     = "typical_misc_storage_chest.png"

print("Samples:", ", ".join(tt.samples()))
upload = tt.upload_widget(); upload

## 4 · Load it and see what TRELLIS sees

In [ ]:
image, NAME = tt.read_image(upload, url=IMAGE_URL, path=IMAGE_PATH, sample=SAMPLE)
prepped = tt.show_input(pipe, image)

## 5 · Generate (about 30–60 s)

`SEED` changes the result for the same image. `SS_*` is the first stage (the coarse shape), `SLAT_*` the second (detail and colour). More steps: slower, a bit cleaner. Higher CFG: sticks closer to the image.

In [ ]:
SEED = 1
PARAMS = dict(ss_steps=12, ss_cfg=7.5, slat_steps=12, slat_cfg=3.0)

out = tt.generate(pipe, prepped, seed=SEED, **PARAMS)
tt.preview(out, NAME, SEED)

## 6 · Export the GLB (`simplify` 0.95 keeps 5% of the triangles; texture 1024 or 2048)

In [ ]:
glb_path = tt.export(out, NAME, SEED, simplify=0.95, texture_size=1024)

## 7 · (Optional) Compare seeds, one still each (about 40 s a seed); then set `SEED` in cell 5

In [ ]:
tt.compare_seeds(pipe, prepped, seeds=[1, 2, 3, 4], **PARAMS)

## 8 · (Optional) Several photos of one object: paths or URLs; TRELLIS's example set when empty

In [ ]:
tt.multi(pipe, [], seed=SEED, **PARAMS)